## MNIST Digit Classification

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 


In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
df1 = train

In [ ]:
df1['label'].value_counts()  # Multiclass Classification

In [82]:
X = df1.drop(columns=['label'])
Y = df1['label']

In [83]:
import sklearn
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size=0.2,random_state=42)

In [84]:
x_train.shape

(33600, 784)

In [85]:
x_train_flat = x_train.to_numpy().reshape(-1,784) /255.0
x_test_flat = x_test.to_numpy().reshape(-1,784) /255.0

### Reducing Dimensions by PCA

In [86]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.99)
x_train_pca = pca.fit_transform(x_train_flat)
x_test_pca = pca.transform(x_test_flat)

In [87]:
x_train_pca.shape

(33600, 330)

## KNN Approach

In [88]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(x_train_pca,y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [89]:
y_pred_train_knn = knn.predict(x_train_pca)
acc_train_knn = accuracy_score(y_train, y_pred_train_knn)
print(acc_train_knn)
y_pred_test_knn = knn.predict(x_test_pca)
acc_test_knn = accuracy_score(y_test, y_pred_test_knn)
print(acc_test_knn)

0.9775892857142857
0.9660714285714286


## Kernel SVM Approach

In [90]:
from sklearn.svm import SVC

svc = SVC(kernel='rbf', C=8)

svc.fit(x_train_pca,y_train)

,C,8
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [91]:
y_pred_train_svc = svc.predict(x_train_pca)
acc_train_svc = accuracy_score(y_train, y_pred_train_svc)
print(acc_train_svc)
y_pred_test_svc = svc.predict(x_test_pca)
acc_test_svc = accuracy_score(y_test, y_pred_test_svc)
print(acc_test_svc)

0.9999702380952381
0.9808333333333333


## CNN Approach

In [6]:
x_train = x_train.to_numpy().reshape((-1,28,28,1))

In [7]:
x_test = x_test.to_numpy().reshape((-1,28,28,1))

#### Data Augmentation

In [43]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers

In [64]:

inputs = keras.Input(shape=(28,28,1))
#x = data_augmentation(inputs)
x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(64, 3, activation='relu', padding='same', kernel_regularizer=regularizers.l2(0.0001))(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(128,3, activation='relu', padding='same', kernel_regularizer=regularizers.l2(0.0002))(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(256,3, activation='relu', padding='same', kernel_regularizer=regularizers.l2(0.0002))(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Dropout(0.25)(x)
x = layers.Flatten()(x)
outputs= layers.Dense(10, activation='softmax')(x)
model = keras.Model(inputs,outputs)


In [66]:
model.compile(optimizer='rmsprop',
              loss = 'sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [67]:
model.fit(x_train,y_train, epochs=10, batch_size=128)

Epoch 1/10
263/263 ━━━━━━━━━━━━━━━━━━━━ 209s 783ms/step - accuracy: 0.9254 - loss: 0.5468
Epoch 2/10
263/263 ━━━━━━━━━━━━━━━━━━━━ 199s 756ms/step - accuracy: 0.9806 - loss: 0.1226
Epoch 3/10
263/263 ━━━━━━━━━━━━━━━━━━━━ 1635s 6s/step - accuracy: 0.9857 - loss: 0.0984
Epoch 4/10
263/263 ━━━━━━━━━━━━━━━━━━━━ 190s 721ms/step - accuracy: 0.9889 - loss: 0.0817
Epoch 5/10
263/263 ━━━━━━━━━━━━━━━━━━━━ 198s 752ms/step - accuracy: 0.9899 - loss: 0.0740
Epoch 6/10
263/263 ━━━━━━━━━━━━━━━━━━━━ 201s 763ms/step - accuracy: 0.9907 - loss: 0.0687
Epoch 7/10
263/263 ━━━━━━━━━━━━━━━━━━━━ 198s 752ms/step - accuracy: 0.9920 - loss: 0.0626
Epoch 8/10
263/263 ━━━━━━━━━━━━━━━━━━━━ 192s 730ms/step - accuracy: 0.9928 - loss: 0.0585
Epoch 9/10
263/263 ━━━━━━━━━━━━━━━━━━━━ 192s 729ms/step - accuracy: 0.9923 - loss: 0.0586
Epoch 10/10
263/263 ━━━━━━━━━━━━━━━━━━━━ 193s 735ms/step - accuracy: 0.9935 - loss: 0.0517


In [68]:
train_loss, train_acc = model.evaluate(x_train, y_train)
print(train_acc)

1050/1050 ━━━━━━━━━━━━━━━━━━━━ 40s 37ms/step - accuracy: 0.9975 - loss: 0.0383
0.9975000023841858


In [69]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(test_acc)

263/263 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - accuracy: 0.9912 - loss: 0.0701
0.991190493106842


In [71]:
df2 = test

In [72]:
df2.head()

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [73]:
df2_test = df2.to_numpy().reshape((-1,28,28,1))

In [74]:
y_prob = model.predict(df2_test)

875/875 ━━━━━━━━━━━━━━━━━━━━ 33s 37ms/step


In [75]:
y_pred = np.argmax(y_prob, axis=1)

In [76]:
ImageId = np.arange(1, len(y_pred) + 1)

In [77]:
submission = pd.DataFrame({'ImageId': ImageId, 'Label':y_pred})

In [78]:
submission.head()

,ImageId,Label
0,1,2
1,2,0
2,3,9
3,4,0
4,5,3


In [79]:
submission.to_csv('submission.csv', index=False)